# Notebook de obtención, limpieza y transformación de datos — Fase 2

**Proyecto transversal · MCDI500 Programación para la Ciencia de Datos**
**Magíster en Ciencia de Datos e Inteligencia Artificial · Universidad Andrés Bello**

**Grupo 5 — Factores asociados a la duración autorizada de los permisos de uso de vía
pública en San Francisco**

## Introducción

Este cuaderno toma el archivo original de permisos y deja un conjunto validado, listo para
modelar la duración autorizada en la Fase 3.

**Lo que recibe de la Fase 1.** `docs/diccionario_variables.csv` y `docs/metadatos_fase1.json`:
qué variable es el objetivo, cuáles son explicativas, cuáles están prohibidas por contener la
respuesta, y cuál agrupa la partición. Esta fase **no vuelve a decidirlo**; lo lee y se detiene
si no lo encuentra.

**Lo que entrega.** `data/processed/permisos_limpio.csv` —interpretable, para el análisis y los
gráficos de la Fase 4—, `data/processed/permisos_procesado.csv` —codificado y escalado, para el
modelado de la Fase 3—, los transformadores ajustados en `src/`, y los anexos del informe en
`docs/anexos/`.

> **Un principio gobierna el orden de los apartados.** La partición va **antes** de cualquier
> ajuste de imputador, codificador o escalador. Calcular una mediana, una moda o una escala
> sobre el conjunto completo filtra al conjunto de prueba información que no debería conocer.
> El orden no es estético: es la diferencia entre una evaluación honesta y una inflada.

### Librerías utilizadas

Cada importación responde a una necesidad concreta. `GroupShuffleSplit` particiona respetando
los grupos; `OneHotEncoder` codifica categorías sin inventar un orden; los tres escaladores se
comparan entre sí en el apartado 5 antes de elegir uno.

In [1]:
import io, contextlib                # capturar salidas para los anexos del informe
import json                          # contrato con la Fase 1 y metadatos
import pickle                        # persistir los transformadores ajustados
import sys
import warnings                      # tratar los avisos de parseo como evidencia
from datetime import date
from pathlib import Path

import numpy as np                   # cálculo numérico y enmascaramiento aleatorio
import pandas as pd                  # estructuras tabulares
import matplotlib.pyplot as plt      # gráficos de exploración y control

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import (OneHotEncoder, StandardScaler, MinMaxScaler,
                                   RobustScaler, LabelEncoder, OrdinalEncoder)

# Reproducibilidad: la misma semilla que el cuaderno de la Fase 1.
SEMILLA = 42
np.random.seed(SEMILLA)


def capturar(funcion, *args, **kwargs):
    """
    Ejecuta una funcion, muestra su salida en pantalla y ademas la devuelve como texto.

    Sirve para que los anexos del informe se generen desde la misma ejecucion que produce
    los resultados, en lugar de copiarse a mano desde una captura de pantalla.
    """
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        resultado = funcion(*args, **kwargs)
    texto = buffer.getvalue()
    print(texto, end="")
    return resultado, texto


print("Cuaderno de la Fase 2 ·", date.today().isoformat())

Cuaderno de la Fase 2 · 2026-09-16


## Configuración del entorno y de las rutas

Las rutas se resuelven igual que en la Fase 1: subiendo hasta encontrar un marcador de
repositorio, en lugar de suponer que la raíz coincide con el directorio desde el que se lanza
el cuaderno. Así ambos cuadernos escriben en el mismo árbol de archivos aunque vivan en
carpetas distintas.

In [3]:
MARCADORES = (".git", "requirements.txt", ".gitignore")


def localizar_raiz(inicio=None, marcadores=MARCADORES, niveles=5, respaldo=True):
    """Resuelve la raiz del proyecto. Identica a la de la Fase 1."""
    actual = Path(inicio or Path.cwd()).resolve()
    for candidata in [actual, *actual.parents][:niveles + 1]:
        if any((candidata / m).exists() for m in marcadores):
            return candidata, "estructurado"
    if respaldo:
        return actual, "plano"
    raise FileNotFoundError(f"No se encontro marcador de repositorio desde {actual}.")


RAIZ, MODO = localizar_raiz()
DIR_CRUDO = RAIZ / "data" / "raw"
DIR_PROCESADO = RAIZ / "data" / "processed"
DIR_DOCS = RAIZ / "docs"
DIR_SRC = RAIZ / "src"
DIR_ANEXOS = DIR_DOCS / "anexos"
for d in (DIR_PROCESADO, DIR_DOCS, DIR_SRC, DIR_ANEXOS):
    d.mkdir(parents=True, exist_ok=True)

# Modulo compartido, generado por el cuaderno de la Fase 1.
if str(DIR_SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(DIR_SRC.resolve()))
try:
    import utilidades
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "No se encontro src/utilidades.py. Ejecute primero el cuaderno de la Fase 1."
    ) from e

print("Raíz del proyecto :", RAIZ)
print("Modo detectado    :", MODO)
print("Módulo importado  :", Path(utilidades.__file__).relative_to(RAIZ).as_posix())

Raíz del proyecto : /Users/ricardo/Desktop/Repositorios/proyecto-grupo5-mcdi500
Modo detectado    : estructurado
Módulo importado  : src/utilidades.py
